

<div style="width:100%;text-align:center;">
    <h1>
        Predicting Nepal earthquake damage on buildings using the tree-based machine learning methods 
    </h1>
    <h3>
        Ibrahim Onur Serbetci
    </h3>
</div>

# Table of Contents
1. [Problem Description](#problem)
1. [Import Libraries](#import-libraries)
1. [Get the Data](#get-the-data)
    1. [Take a quick look at data structure](#quick-look)
    1. [Create a test set](#create-test)
1. [Exploratory Data Analysis](#eda)
    1. [Description of Features](#desc)
    1. [Discover and Visualize](#graph)
    1. [Looking for Correlations](#cor)
    1. [Insights from Data Analysis](#eda-res)
1. [Feature Engineering](#feature)
1. [Machine Learning](#ml)
    1. [Decision Tree Classifier](#dt)
        1. [Fine Tune Decision Trees](#dt-f)
    1. [Random Forest Classifier](#rf)
        1. [Fine Tune Random Forest](#rf-f)
    1. [Comparison](#comparison)
1. [References](#ref)

<a name='problem'></a>
# Problem Description

<script src="https://polyfill.io/v3/polyfill.min.js?features=es6"></script>
<script type="text/javascript" id="MathJax-script" async
  src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js">
</script>
        
Every year The National Earthquake Information Center (NEIC) records an average of 20,000 earthquakes all around to world. This number is fairly huge, but we can infer from that in order to understand more that is huge, averagely 50 earthquake occured in each day$^{[1]}$.


In 2015 April 25, an intense earthquake occured in Central Nepal at local time of 11:56 a.m. My goal is the predict level of damage of the buildings in 2015 Gorkha earthquake in Nepal. The data that was collected through surveys by [Kathmandu Living Labs](http://www.kathmandulivinglabs.org/) and the [Central Bureau of Statistics](https://cbs.gov.np/) will be used in this notebook. This data one of the largest post-disaster dataset ever collected, it includes various information that influence various aspects of science.

**Note:** The triple dots in the below means there are some hidden codes. It can be opened by clicking on them. This is same for all triple dots in the notebook. (If there is not triple dots or no code hidden, do not consider this note)

In [ ]:
from IPython.display import Image, HTML
Image(url='https://s3.amazonaws.com/drivendata-public-assets/nepal-quake-bm-2.JPG')

You can inspect full data from the [link](http://eq2015.npc.gov.np/)

<a name='import-libraries'></a>
# Import Libraries

Various libraries have been imported for various tasks. These are,
* Create data frame
    * [Pandas](https://pandas.pydata.org/)
* Linear Algebra or math operations
    * [Numpy](https://numpy.org/)
    * [Math](https://docs.python.org/3/library/math.html)
* Data Visualization
    * [Matplotlib](https://matplotlib.org/)
    * [Plotly](https://plotly.com/)
    * [Seaborn](https://seaborn.pydata.org/)
* Statistics
    * [Scipy](https://www.scipy.org/)
* Machine Learning
    * [Scikit-learn](https://scikit-learn.org/stable/)
        * Train test Split:
            * [model_selection.train_test_split()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
        * Feature Engineering and preprocessing:
            * [preprocessing.LabelEncoder()](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html)
            * [preprocessing.OneHotEncoder()](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
            * [preprocessing.scale()](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.scale.html)
            * [compose.ColumnTransformer()](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html)
            * [decomposition.PCA()](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
        * Fitting the model and evaluation:
            * [tree.DecisionTreeClassifier()](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
            * [ensemble.RandomForestClassifier()](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
            * [metrics.classification_report()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
            * [metrics.confusion_matrix()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
            * [metrics.accuracy_score()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
        * Fine Tuning the hyperparameter:
            * [model_selection.RandomizedSearchCV()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
            * [model_selection.cross_val_score()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
            * [model_selection.GridSearchCV()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
            * [model_selection.validation_curve()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.validation_curve.html)
            * [model_selection.StratifiedKFold()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)

In [ ]:
# Create DataFrame
import pandas as pd

# Math transformations
import numpy as np
import math

# Data Viz
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode, iplot, plot

# Stats
from scipy import stats
from scipy.stats import randint

# Train test split
from sklearn.model_selection import train_test_split

# Feature engineering
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, scale
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

# Machine Learning
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Fine-tune
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.model_selection import GridSearchCV, validation_curve, StratifiedKFold

<a name='get-the-data'></a>
# Get the data

The data can be imported from `Amazon AWS Storage`. This version of data provided by `drivendata` for educational competetion purposes. In order to import data from the urls, `pandas.read_csv()` function can be used. Data divided into two part, features and target values for each building. It will be investigate both features and target values in next sections. 

In [ ]:
features = pd.read_csv('https://s3.amazonaws.com/drivendata/data/57/public/train_values.csv').set_index('building_id')
target = pd.read_csv('https://s3.amazonaws.com/drivendata/data/57/public/train_labels.csv').set_index('building_id')

It will be easier to aggregate `features` and `target` for future operations. `pandas.concat()` function could be used in order to do it.   

In [ ]:
df=pd.concat([features,target],axis=1)
display(HTML('<h3>A picture from aggregated DataFrame:</h3> <br>'))
display(df.head())

<a name='quick-look'></a>
## Take a Quick Look at Data Structures

It will be much easier to look at the data before [Exploratory Data Analysis](#eda). It may be necessary check the how much space it is taken by the data.

In [ ]:
display(HTML('<h4>Memory usage of each attribute in megabytes:</h4><br>'))
df.memory_usage(deep=True)/(10**6)

In [ ]:
display(HTML(f'<h4>Shape of the data: {df.shape}</h4>'))

It looks like such a big data, automating some steps would help the process.

<a name='create-test'></a>
## Create a test set

Randomly selected 80% of the data will be used for [Exploratory Data Analysis](#eda) section. This will be prevent the concept of data snooping$^{[2]}$.

In [ ]:
eda,_ = train_test_split(df, test_size=.2, random_state=42)

<a name='eda'></a>
# Explatory Data Analysis

<a name='desc'></a>
## Description of Features
* **geo_level_1_id, geo_level_2_id, geo_level_3_id (type: int):** geographic region in which building exists, from largest (level 1) to most specific sub-region (level 3). Possible values: level 1: 0-30, level 2: 0-1427, level 3: 0-12567.

* **count_floors_pre_eq (type: int):** number of floors in the building before the earthquake.

* **age (type: int):** age of the building in years.

* **area_percentage (type: int):** normalized area of the building footprint.

* **height_percentage (type: int):** normalized height of the building footprint.

* **land_surface_condition (type: categorical):** surface condition of the land where the building was built. Possible values: n, o, t.

* **foundation_type (type: categorical):** type of foundation used while building. Possible values: h, i, r, u, w.

* **roof_type (type: categorical):** type of roof used while building. Possible values: n, q, x.

* **ground_floor_type (type: categorical):** type of the ground floor. Possible values: f, m, v, x, z.

* **other_floor_type (type: categorical):** type of constructions used in higher than the ground floors (except of roof). Possible values: j, q, s, x.

* **position (type: categorical):** position of the building. Possible values: j, o, s, t.

* **plan_configuration (type: categorical):** building plan configuration. Possible values: a, c, d, f, m, n, o, q, s, u.

* **has_superstructure_adobe_mud (type: binary):** flag variable that indicates if the superstructure was made of Adobe/Mud.

* **has_superstructure_mud_mortar_stone (type: binary):** flag variable that indicates if the superstructure was made of Mud Mortar - Stone.

* **has_superstructure_stone_flag (type: binary):** flag variable that indicates if the superstructure was made of Stone.

* **has_superstructure_cement_mortar_stone (type: binary):** flag variable that indicates if the superstructure was made of Cement Mortar - Stone.

* **has_superstructure_mud_mortar_brick (type: binary):** flag variable that indicates if the superstructure was made of Mud Mortar - Brick.

* **has_superstructure_cement_mortar_brick (type: binary):** flag variable that indicates if the superstructure was made of Cement Mortar - Brick.

* **has_superstructure_timber (type: binary):** flag variable that indicates if the superstructure was made of Timber.

* **has_superstructure_bamboo (type: binary):** flag variable that indicates if the superstructure was made of Bamboo.

* **has_superstructure_rc_non_engineered (type: binary):** flag variable that indicates if the superstructure was made of non-engineered reinforced concrete.

* **has_superstructure_rc_engineered (type: binary):** flag variable that indicates if the superstructure was made of engineered reinforced concrete.

* **has_superstructure_other (type: binary):** flag variable that indicates if the superstructure was made of any other material.

* **legal_ownership_status (type: categorical):** legal ownership status of the land where building was built. Possible values: a, r, v, w.

* **count_families (type: int):** number of families that live in the building.

* **has_secondary_use (type: binary):** flag variable that indicates if the building was used for any secondary purpose.

* **has_secondary_use_agriculture (type: binary):** flag variable that indicates if the building was used for agricultural purposes.

* **has_secondary_use_hotel (type: binary):** flag variable that indicates if the building was used as a hotel.

* **has_secondary_use_rental (type: binary):** flag variable that indicates if the building was used for rental purposes.

* **has_secondary_use_institution (type: binary):** flag variable that indicates if the building was used as a location of any institution.

* **has_secondary_use_school (type: binary):** flag variable that indicates if the building was used as a school.

* **has_secondary_use_industry (type: binary):** flag variable that indicates if the building was used for industrial purposes.

* **has_secondary_use_health_post (type: binary):** flag variable that indicates if the building was used as a health post.

* **has_secondary_use_gov_office (type: binary):** flag variable that indicates if the building was used fas a government office.

* **has_secondary_use_use_police (type: binary):** flag variable that indicates if the building was used as a police station.

* **has_secondary_use_other (type: binary):** flag variable that indicates if the building was secondarily used for other purposes.

Profiling the data will be help for studying each feature.

In [ ]:
# !pip3 install 'pandas-profiling[notebook]'
# from pandas_profiling import ProfileReport

# profile = ProfileReport(eda,)
# profile.to_notebook_iframe()

After reading all the report, some interesting piece will be investigate in next section.

<a name='graph'></a>
## Discover and Visualize

In this section, some of questions will be discussed briefly in order to analyse the data and get to know. Some of the basic questions will be answered in this chapter which are as follows. 
* What is distribution of damages on all buildings?
* How the age of the building affects damage due to earthquakes?
* What is the distribution of area of the buildings affected by damage due to earthquakes?

In [ ]:
# This transformation needs in order to avoid confusion of damage_grade variable as continous
pd.set_option('mode.chained_assignment', None)
eda['damage_grade(eda)'] = np.where(eda.damage_grade==1,'(1) Low', 
                                    np.where(eda.damage_grade==2,'(2) Medium', 
                                             np.where(eda.damage_grade==3,'(3) High',0)))

* ***Investigate target variable (Damage Grade)***

In [ ]:
fig = px.pie(eda.groupby('damage_grade(eda)').age.count().reset_index(), values='age', names='damage_grade(eda)' , title='Damage Grade Distribution')
fig.show()

- ***OBSERVATIONS***
    - 1: represents low damage
    - 2: represents a medium amount of damage
    - 3: represents almost complete destruction
    - `9.57%` of bulidings were less damaged by earthquake.
    - `57%` of bulidings were medium damaged 
    - `33.5%` of bulidings were highly damaged due to earthquake.

* ***Age of building vs Damage***

In [ ]:
fig = px.bar(eda.groupby(['age','damage_grade(eda)']).roof_type.count().reset_index().rename(columns={'roof_type':'count'}),
             x="age", y="count", color="damage_grade(eda)", title="Check if the age of the building affect on damage due to earthquake?")
fig.update_xaxes(range=[0, 100])
fig.show()

- ***OBSERVATIONS***
    - Around 90% of building ages fall under \[0-50] range
    - 2nd highest no. of bulidings are in the category high damage.
    - Inferences, can be seen by pie chart below more easily.

In [ ]:
def draw_subplotted_pie_chart(eda ,col, labels, target_col):
    pd.set_option('mode.chained_assignment', None)    

    fig = make_subplots(rows=1, cols=4, specs=[[{'type':'domain'}, 
                                                {'type':'domain'},
                                                {'type':'domain'},
                                                {'type':'domain'}]], subplot_titles = labels)
    for i,lb in enumerate(labels):
        eda_labeled = eda[eda[target_col]==lb]
        eda_counted = pd.DataFrame(eda_labeled.groupby(col)[col].count()).rename(columns={
            col:'Count'}).reset_index()
        fig.add_trace(go.Pie(values=eda_counted.Count, labels=eda_counted[col], name=lb),1,i+1)
    fig.update_layout(title_text=f'{col} distribution for each {target_col}')
    iplot(fig)

labels=['0-10','10-15','15-30','30-995']
eda['age_cut']=pd.qcut(eda.age,4,labels=labels)
draw_subplotted_pie_chart(eda,'damage_grade(eda)',labels, 'age_cut')

- ***FURTHER OBSERVATIONS***
    - The age is making difference on damage grade until 15.


* ***Area of building vs Damage***

In [ ]:
import warnings
warnings.filterwarnings('ignore')
sns.FacetGrid(eda,hue='damage_grade',height=5,aspect=2,palette="viridis")\
    .map(sns.distplot,'area_percentage')\
    .add_legend()
plt.title("Area Percentage")

<a name='corr'></a>
## Looking for Correlations

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(10,8))
cor=eda.corr()
cor=pd.DataFrame(cor)
sns.heatmap(cor,cmap="viridis",ax=ax)

<a name='eda-res'></a>
## Insights from Data Analysis

* Remove `building_id` in order to avoid misguide the model
* There are so many outliers, however, Random Forests algorithm is robust to the outliers$^{[3]}$
* Should change encoding system with `LabelEncoder` or `OneHotEncoder` considering whether each feature is *incremental* or not.
* There isn't any missing values in the dataset
* Since binary variable not belong to coordinate system `PCA` is failing to reducing dimensions of the dataset. Thus, this step will be skipped.$^{[4]}$ 

<a name='feature'></a>
# Feature Engineering

* ***Basic Engineering***

There is no incremental feature in the dataset. Therefore, `OneHotEncoder` is sufficient for feature engineering.

In [ ]:
df_pre = df.reset_index().copy()

# Select data from datatype of each attr which is other than numbers and apply encoder
df_enc=df_pre.select_dtypes(exclude=["number"])
enc = OneHotEncoder(sparse=False,dtype=np.int64)
enc_matrix=enc.fit_transform(df_enc)

# Concat the encoded matrix with the other ones
df_basic = pd.concat([pd.DataFrame(enc_matrix, 
                                   columns=enc.get_feature_names(list(df_enc))),
                      df_pre],
                     axis=1).drop(list(df_enc), axis=1)

# Divide dataset by target and feature variables
y = df_basic.loc[:,'damage_grade']
X = df_basic.drop(['damage_grade','building_id'], axis=1)

<a name='ml'></a>
# Machine Learning

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2,
                                                    random_state=42, stratify=y)
cv = StratifiedKFold(5, random_state=42)

<a name='dt'></a>
## Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
cvs_res=cross_val_score(dt, X, y, cv=cv, scoring="accuracy", n_jobs=-1, verbose=5)
print(f'\nCross validated accuracy scores: {cvs_res}\n\n')
dt.fit(X_train, y_train)

dt_predictions=dt.predict(X_test)
print(f'accuracy_score: {accuracy_score(y_test, dt_predictions)}\n\n')
print(classification_report(y_test, dt_predictions))

<a name='dt-f'></a>
### Fine Tune Decision Tree Classifier

In [ ]:

%%time
param_dist = {
              "max_depth": np.linspace(75,125,5, dtype=np.int64),
              "max_leaf_nodes": np.linspace(2000,2500,10, dtype=np.int64),
    
}
tree_cv = GridSearchCV(DecisionTreeClassifier(random_state=42), param_dist, cv=5, n_jobs=-1, verbose=10)
tree_cv.fit(X,y)
print("Tuned Decision Tree Parameters: {}".format(tree_cv.best_params_))
print("Best score is {}".format(tree_cv.best_score_))

In [ ]:
def plot_cm(cf_matrix, save=False):
    labels=['Grade 1','Grade 2','Grade 3']
    sns.heatmap(cf_matrix/np.sum(cf_matrix), annot=True, xticklabels=labels, yticklabels=labels,
            fmt='.2%')
    if save:
        plt.savefig('best_dt_conf.png')

In [ ]:
dt_best = DecisionTreeClassifier(max_depth=75, max_leaf_nodes=2333, random_state=42)
dt_best.fit(X_train,y_train)
dt_predictions=dt_best.predict(X_test)
print(f'accuracy_score: {accuracy_score(y_test, dt_predictions)}\n\n')
print(classification_report(y_test, dt_predictions))
cf_matrix = confusion_matrix(y_test, dt_predictions)
plot_cm(cf_matrix)

<a name='rf'></a>
## Random Forest Classifier

In [ ]:
%%time
rf = RandomForestClassifier(random_state=42, n_jobs=-1, oob_score=True)
rf.fit(X_train, y_train)

cvs_res=cross_val_score(rf, X, y, cv=cv, scoring="accuracy", n_jobs=-1, verbose=5)
print(f'\nCross validated accuracy scores: {cvs_res}\n\n')

rf_predictions=rf.predict(X_test)
print(f'\naccuracy_score: {accuracy_score(y_test, rf_predictions)}\n\n')
print(classification_report(y_test, rf_predictions))

<a name='rf-f'></a>
### Fine Tune Random Forest Classifier

In [ ]:
def plot_oob(acc,oob,parameters):
    rf_best_oob=pd.DataFrame(parameters,columns=['Number of Tree'])
    rf_best_oob['Accuracy Score'],rf_best_oob['Out-of-bag Score'] = acc,oob
    dd=rf_best_oob[['Accuracy Score','Out-of-bag Score']].stack().reset_index().rename({'level_0':'Number of Tree',
                                                                                        'level_1':'Score Type',
                                                                                        0:'Score'},axis=1)
    dd=dd.replace({'Number of Tree':{key:value for (key,
                                                    value) in zip(np.arange(0,6,1),
                                                                  rf_best_oob[['Number of Tree']].values)}})
    fig=px.line(dd, x='Number of Tree', y='Score', color='Score Type', labels={'index':'Fold ID'})
    fig.update_xaxes(type='category')
    fig.show()

In [ ]:
%%time
acc,oob=[],[]
for i in [10, 100, 300, 500, 700, 900]:
    rf = RandomForestClassifier(n_estimators=i, random_state=42, n_jobs=-1, oob_score=True,verbose=1)
    rf.fit(X_train, y_train)
    rf_predictions=rf.predict(X_test)
    acc.append(accuracy_score(y_test, rf_predictions))
    oob.append(rf.oob_score_)

# Plot
plot_oob(acc,oob,[10, 100, 300, 500, 700, 900])

In [ ]:
%%time
rf_best = RandomForestClassifier(n_estimators=100, random_state=42)
rf_best.fit(X_train,y_train)
rf_predictions=rf_best.predict(X_test)
print(f'accuracy_score: {accuracy_score(y_test, rf_predictions)}\n\n')
print(classification_report(y_test, rf_predictions))
cf_matrix = confusion_matrix(y_test, rf_predictions)
plot_cm(cf_matrix)

<a name='comparison'></a>
## Comparison

In [ ]:
%%time
dt_f = DecisionTreeClassifier(random_state = 24, max_depth=75, max_leaf_nodes=2333)
dt_f.fit(X_train,y_train)
dt_f_pred=dt_f.predict(X_test)

rf_f = RandomForestClassifier(n_estimators=100, random_state = 24)
rf_f.fit(X_train,y_train)
rf_f_pred=rf_f.predict(X_test)

# Classification Report
display(HTML('<h4>Decision Trees Classification Report</h4>'))
print(classification_report(y_test, dt_f_pred))
display(HTML('<h4>Random Forest Classification Report</h4>'))
print(classification_report(y_test, rf_f_pred))

#Confusion Matrix
cm_dt=confusion_matrix(y_test,dt_f_pred)
conf_matrix_dt=pd.DataFrame(data=cm_dt,columns=['Predicted:1','Predicted:2','Predicted:3'],
                                         index=['Actual:1','Actual:2','Actual:3'])
cm_rf=confusion_matrix(y_test,rf_f_pred)
conf_matrix_rf=pd.DataFrame(data=cm_rf,columns=['Predicted:1','Predicted:2','Predicted:3'],
                                         index=['Actual:1','Actual:2','Actual:3'])


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,7), sharey=True)
fig.suptitle('Comparison of Decision Tree and Random Forests Classifiers')
sns.heatmap(conf_matrix_dt, annot=True,fmt='d',cmap="rocket_r",ax=ax1, cbar=False)
sns.heatmap(conf_matrix_rf, annot=True,fmt='d',cmap="rocket_r",ax=ax2)

<a name='ref'></a>
# References

**[1]** FEDERAL EMERGENCY MANAGEMENT AGENCY, Semisonic Sleuths. AMERICAN GEOPHYSICAL UNION, Accessed December 20, 2020, http://www.fema.gov/media-library-data/20130726-1646-20490-4697/fema253.pdf.

**[2]** A. Géron, Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow. O'Reilly, 2019.

**[3]** L. Breiman, "Random Forests", Machine Learning, vol. 45, no. 1, pp. 5-32, 2001. Available: 10.1023/a:1010933404324 [Accessed 10 January 2021].

**[4]** B. Walker, "PCA Is Not Feature Selection", Towards Data Science, 2019. .